In [ ]:
# !pip install -q langchain==0.1.20 langchain-openai chromadb pypdf sentence_transformers tiktoken langchain-community

# 1. 에러를 유발하는 simsimd 삭제 및 안정적인 버전 재설치
!pip uninstall -y simsimd
!pip install -q "langchain==0.1.20" "langchain-openai" "chromadb" "pypdf" "tiktoken" "langchain-community==0.0.38" "numpy<2.0.0"

Found existing installation: simsimd 6.5.16
Uninstalling simsimd-6.5.16:
  Successfully uninstalled simsimd-6.5.16


In [ ]:
# 2. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. API 키 설정 (여기에 OpenAI API 키를 입력하세요)
import os
os.environ['OPENAI_API_KEY'] = 'sk'

In [ ]:
# 4. 모듈 임포트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
import tiktoken

In [ ]:
# 5. 토크나이저 설정
tokenizer = tiktoken.get_encoding("cl100k_base")   # GPT-4, GPT-3.5-turbo 모델들이 사용하는 인코딩 방식

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

In [ ]:
# 6. PDF 로드 및 분할
# 파일 경로가 맞는지 다시 한번 확인해주세요.
loader = PyPDFLoader("/content/drive/MyDrive/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=50,
    length_function=tiktoken_len
)
texts = text_splitter.split_documents(pages)

In [ ]:
# 7. 임베딩 및 벡터 스토어 생성
# HuggingFace 대신 OpenAI의 임베딩 모델을 사용합니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
# 벡터 스토어 생성
docsearch = Chroma.from_documents(texts, embedding_model)

In [ ]:
# 8. OpenAI LLM 설정 (GPT-4o 또는 GPT-3.5-turbo 등)
llm_openai = ChatOpenAI(model="gpt-4o", temperature=0.0)

In [ ]:
# 9. RetrievalQA 체인 생성
qa = RetrievalQA.from_chain_type(
    llm=llm_openai,
    chain_type="stuff",
    retriever=docsearch.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10}
    )
)

In [ ]:
# 질문 실행 (이제 정상 작동할 것입니다)
query = "how demian looks like"
try:
    result = qa.invoke(query)
    from IPython.display import Markdown, display
    display(Markdown(result["result"]))
except TypeError as e:
    print(f"라이브러리 호환성 에러 발생: {e}")
    print("해결책: '!pip uninstall -y simsimd' 실행 후 런타임을 다시 시작하세요.")

싱클레어는 헤르만 헤세의 소설 "데미안"의 주인공입니다. 그는 소설에서 자신의 정체성과 자아를 찾아가는 과정을 겪으며, 여러 인물들과의 관계를 통해 성장하고 변화하는 모습을 보여줍니다. 싱클레어는 특히 프란츠 크로머와의 갈등, 그리고 피스토리우스와의 대화를 통해 많은 것을 배우고 깨닫게 됩니다.